In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [2]:
## 2维互相关运算
def corr2d(X,K):
    h,w = K.shape ## the shape of convolution kernel
    Y = torch.zeros((X.shape[0]-h+1,X.shape[1]-w+1)) ## 初始化输出
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i,j] = (X[i:i+h,j:j+w]*K).sum()

    return Y

X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X, K)

tensor([[19., 25.],
        [37., 43.]])

In [3]:
## convolution layer
class Con2d(nn.Module):
    def __init__(self,kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))
    def forward(self,x):
        return corr2d(x,self.weight) + self.bias

In [4]:
## 应用卷积层识别黑白图像边缘
X = torch.ones((6,8))
X[:,2:6] = 0

K = torch.tensor([[1.0,-1.0]])

Y = corr2d(X,K)
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

In [5]:
## learning convolution kernel
conv2d = nn.Conv2d(1,1,kernel_size=(1,2),bias=False)

X = X.reshape((1,1,6,8))
Y = Y.reshape((1,1,6,7))
lr = 3e-2

for i in range(10):
    Y_hat = conv2d(X)
    l = (Y_hat-Y)**2
    conv2d.zero_grad()
    l.sum().backward()
    conv2d.weight.data[:] -= lr*conv2d.weight.grad
    if (i + 1) % 2 == 0:
        print(f'epoch {i+1}, loss {l.sum():.3f}')
conv2d.weight.data.reshape((1,2))

epoch 2, loss 12.898
epoch 4, loss 4.249
epoch 6, loss 1.567
epoch 8, loss 0.613
epoch 10, loss 0.246


tensor([[ 0.9384, -1.0401]])

In [6]:
## multi input channels
def multi_in(X,K):
    return sum(d2l.corr2d(x,k) for x,k in zip(X,K))

X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
[[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])
multi_in(X, K)

tensor([[ 56.,  72.],
        [104., 120.]])

In [7]:
## multi input_output channels
def corr2d_multi_in_out(X, K):
    # 遍历 K 的第 0 维（输出通道），每次取出一个形状为 (C_in, k_h, k_w) 的卷积核 k
    # 对 X 和 k 做多输入单输出互相关，得到形状为 (H_out, W_out) 的特征图
    # 最后将所有输出通道的结果在第 0 维堆叠，得到 (C_out, H_out, W_out)
    return torch.stack([multi_in(X, k) for k in K], 0)



In [8]:
K = torch.stack((K, K + 1, K + 2), 0)
K.shape

corr2d_multi_in_out(X,K)

tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])

In [9]:
## 1*1 conv kernel的作用
def corr2d_multi_in_out_1x1(X,K):
    c_i,h,w = X.shape
    c_o = K.shape[0]
    ## 1*1 kernel下，各个像素之间独立，可以直接flatten
    X = X.reshape((c_i,h*w))
    K = K.reshape((c_o,c_i))
    Y = torch.matmul(K,X)
    return Y.reshape((c_o,h,w))

In [10]:
X = torch.normal(0, 1, (3, 3, 3))
K = torch.normal(0, 1, (2, 3, 1, 1))
Y1 = corr2d_multi_in_out_1x1(X, K)
Y2 = corr2d_multi_in_out(X, K)
Y1

tensor([[[-0.3770,  0.0181, -0.0282],
         [ 2.6367,  0.2346,  0.1630],
         [ 3.9539,  1.0409, -2.1531]],

        [[-3.4637, -1.3086,  2.0315],
         [-2.2457, -0.6756,  2.3278],
         [-1.9718, -6.3279,  5.1003]]])

In [13]:
## pooling layer
def pool2d(X,pool_size,mode='max'):
    p_h,p_w = pool_size
    Y = torch.zeros((X.shape[0]-p_h+1,X.shape[1]-p_w+1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            if mode == 'max':
                Y[i,j] = X[i:i+p_h,j:j+p_w].max()
            elif mode == 'avg':
                Y[i,j] = X[i:i+p_h,j:j+p_w].mean()
    return Y
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
pool2d(X, (2, 2))
pool2d(X, (2, 2), 'avg')

tensor([[2., 3.],
        [5., 6.]])

In [15]:
X = torch.arange(16, dtype=torch.float32).reshape((1, 1, 4, 4))
X

tensor([[[[ 0.,  1.,  2.,  3.],
          [ 4.,  5.,  6.,  7.],
          [ 8.,  9., 10., 11.],
          [12., 13., 14., 15.]]]])

In [17]:
## 默认stride和pool_size一样
pool2d = nn.MaxPool2d(3)
pool2d(X)

pool2d = nn.MaxPool2d(3, padding=1, stride=2)
pool2d(X)

tensor([[[[ 5.,  7.],
          [13., 15.]]]])

In [19]:
X = torch.cat((X, X + 1), 1)
pool2d = nn.MaxPool2d(3, padding=1, stride=2)
pool2d(X)

tensor([[[[ 5.,  7.],
          [13., 15.]],

         [[ 6.,  8.],
          [14., 16.]],

         [[ 6.,  8.],
          [14., 16.]],

         [[ 7.,  9.],
          [15., 17.]]]])